# Evaluating LM Outputs using Rubric + LM Judges

Having issues with running `asyncio` code in Jupyter notebook. Will switch to .py file.

In [2]:
import ast
import datetime
import json
import os
import os.path as osp
from typing import List
import click
import yaml
from blade_bench.eval.datamodel.submission import DatasetSubmission
from blade_bench.logger import logger, formatter
from stat_genie.blade_pipeline.baselines.config import EvalConfig
from blade_bench.eval.datamodel.multirun import MultiRunResults
from stat_genie.blade_pipeline.eval.evaluator import run_eval_on_analyses
from blade_bench.data.datamodel.transforms import (
    TransformDataReturn,
)  # ❗️ this import needs to be kept here for the eval code to work
from stat_genie.blade_pipeline.utils import get_absolute_dir

In [3]:
from stat_genie.blade_pipeline.run_files.run_get_eval import load_list_int

In [4]:
multirun_load_path = "analysis_output/multirun_analyses.json"
llm_eval_config_path = "../../config/llm_eval_config.yml"
output_dir = "eval_output"
submission_load_path = None
cache_code_results = True
diversity_ks = "[]"
diversity_n_samples = 1000

In [5]:
diversity_ks = load_list_int(diversity_ks)
llm_eval_config = yaml.safe_load(open(llm_eval_config_path))
if multirun_load_path:
    with open(multirun_load_path, "r") as f:
        multirun_results = MultiRunResults(**json.load(f))
    dataset_name = multirun_results.dataset_name
    n = multirun_results.n
    logger.info(f"Loaded multirun results from: {multirun_load_path}")
else:

    dataset_submission = DatasetSubmission(
        **json.load(open(submission_load_path, "r"))
    )
    dataset_name = dataset_submission.dataset_name
    n = len(dataset_submission.analyses)
    logger.info(f"Loaded dataset submission from: {submission_load_path}")

[2025-10-31 02:06:25.75][1822240753.py:8 - __main__:<module>][INFO] Loaded multirun results from: analysis_output/multirun_analyses.json


In [6]:
if not output_dir:
    current_time = datetime.datetime.now()
    time_str = current_time.strftime("%Y-%m-%d_%H-%M-%S")
    output_dir = f"./outputs/eval/{dataset_name}_nruns={n}_{time_str}"
if not osp.exists(output_dir):
    os.makedirs(output_dir)

In [7]:
# remove old file
if osp.exists(osp.join(output_dir, "run_eval.log")):
    os.remove(osp.join(output_dir, "run_eval.log"))
logger.add(osp.join(output_dir, f"run_eval.log"), format=formatter.format)

2

In [8]:
command_string = f"python ../../stat_genie/blade_pipeline/run_files/run_get_eval.py "
if multirun_load_path:
    command_string += (
        f"\\\n\t--multirun_load_path {get_absolute_dir(multirun_load_path)} "
    )
else:
    command_string += (
        f"\\\n\t--submission_load_path {get_absolute_dir(submission_load_path)} "
    )

command_string += (
    f"\\\n\t--llm_eval_config_path {get_absolute_dir(llm_eval_config_path)} "
)
if not cache_code_results:
    command_string += "\\\n\t--no_cache_code_reuslts "
command_string += f"\\\n\t--output_dir {get_absolute_dir(output_dir)} "
command_string += f"\\\n\t--ks '{diversity_ks}' "
command_string += f"\\\n\t--diversity_n_samples {diversity_n_samples}"
logger.info(f"Running command: \n{command_string}")
with open(osp.join(output_dir, "command.sh"), "w") as f:
    f.write("""#!/bin/bash\n""")
    f.write(command_string)

logger.info(f"Running evaluation for dataset {dataset_name} with nruns={n}")

[2025-10-31 02:06:25.81][1466812239.py:19 - __main__:<module>][INFO] Running command: 
python ../../stat_genie/blade_pipeline/run_files/run_get_eval.py \
	--multirun_load_path /accounts/grad/zachrewolinski/research/stat-genie/examples/analysis_and_eval/analysis_output/multirun_analyses.json \
	--llm_eval_config_path /accounts/grad/zachrewolinski/research/stat-genie/config/llm_eval_config.yml \
	--output_dir /accounts/grad/zachrewolinski/research/stat-genie/examples/analysis_and_eval/eval_output \
	--ks '[]' \
	--diversity_n_samples 1000
[2025-10-31 02:06:25.82][1466812239.py:24 - __main__:<module>][INFO] Running evaluation for dataset hurricane with nruns=2


In [9]:
eval_config = EvalConfig(
    multirun_load_path=multirun_load_path,
    dataset_submission_path=submission_load_path,
    llm_eval=llm_eval_config,
    output_dir=output_dir,
    use_code_cache=cache_code_results,
    diversity_ks=diversity_ks,
    diversity_n_samples=diversity_n_samples,
)

logger.info(
    f"Running evaluation with config: {eval_config.model_dump_json(indent=2)}"
)
run_eval_on_analyses(eval_config)

[2025-10-31 02:06:25.83][895901646.py:11 - __main__:<module>][INFO] Running evaluation with config: {
  "glob_str": null,
  "multirun_load_path": "analysis_output/multirun_analyses.json",
  "dataset_submission_path": null,
  "llm_eval": {
    "provider": "openai",
    "model": "gpt-5-mini",
    "textgen_config": null,
    "log_file": null,
    "use_cache": true
  },
  "output_dir": "eval_output",
  "run_dataset": null,
  "use_code_cache": true,
  "diversity_ks": [],
  "diversity_n_samples": 1000
}
[2025-10-31 02:06:25.88][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


RuntimeError: asyncio.run() cannot be called from a running event loop